# Klasifikasi Huruf Bahasa Isyarat Amerika (ASL A–Z) dengan Transfer Learning

**Project UAS Deep Learning** — notebook ini mencakup pemuatan data, audit duplikat, pembagian data tanpa kebocoran, pelatihan dua tahap (ekstraksi fitur lalu *fine-tuning*) pada empat model *pretrained* ditambah CNN dari nol, serta evaluasi dan perbandingan model.

**Cara memakai**
1. Aktifkan `.venv` dan pilih kernel "Python (asl-env)". Pelatihan memakai GPU laptop (RTX 2060).
2. Buka sel **Konfigurasi**. Dengan `DATA_SOURCE = "local"`, gambar dibaca dari `./data`.
3. Jalankan seluruh notebook dengan `SMOKE_TEST = True` terlebih dahulu (sekitar 10–15 menit, termasuk mengunduh bobot). Ini hanya memastikan semua sel berjalan tanpa error.
4. Ubah `SMOKE_TEST = False`, lalu jalankan ulang untuk pelatihan penuh. Hasil disimpan di `outputs/full/` per model, sehingga jika proses terputus, model yang sudah selesai akan dilewati saat dijalankan ulang.

**Urutan bagian:** 1 Setup · 2 Konfigurasi · 3 Memuat data · 4 Audit duplikat · 5 Pembagian data · 6 Dataset dan augmentasi · 7 Model · 8 Pelatihan · 9 Menjalankan eksperimen · 10 Kurva pelatihan · 11 Evaluasi sesi yang sudah dikenal · 12 Perbandingan model · 13 Uji pada orang baru · 14 Analisis akhir · 15 Uji foto sendiri (opsional)

## 1. Setup

In [ ]:
import os, sys, json, time, random, hashlib, shutil, zipfile, copy, subprocess, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

try:
    from IPython.display import display
except ImportError:
    display = print

IN_COLAB = "google.colab" in sys.modules
print("Python", sys.version.split()[0])

## 2. Konfigurasi
Semua pengaturan ada di sel ini. Nilai awal mengikuti rencana pada proposal.

In [ ]:
# ---------------- Mode eksekusi ----------------
SMOKE_TEST = False       # True: uji cepat (sedikit gambar, 1-2 epoch). Ubah ke False untuk pelatihan penuh.
SMOKE_PER_SOURCE = 10    # jumlah gambar per huruf per sumber (Kaggle/tiap anggota) pada mode uji cepat

# ---------------- Sumber data ----------------
DATA_SOURCE  = "local"        # "local" (laptop) | "kagglehub" | "drive_zip" | "drive_folder"
KAGGLE_SLUG  = "iamavinashkr090502/asl-hand-landmark-and-gesture-dataset"   # isi dari URL: kaggle.com/datasets/<PENGUNGGAH>/<NAMA-DATASET>
DRIVE_ZIP    = "/content/drive/MyDrive/UAS_DL_ASL/american_hand_sign.zip"
DRIVE_FOLDER = "/content/drive/MyDrive/UAS_DL_ASL/data"
LOCAL_DIR    = "./data"       # folder berisi A/ ... Z/ (boleh bersarang); relatif terhadap lokasi notebook
WORK_DIR     = "/content/drive/MyDrive/UAS_DL_ASL/outputs" if IN_COLAB else "./outputs"

# ---------------- Audit dan pembagian data ----------------
DROP_EXACT_DUPLICATES = True   # gambar identik (MD5 sama) dan labelnya sama: simpan satu salinan
NEAR_DUP_THRESHOLD = 6         # hanya untuk laporan audit di bagian 4; pembagian data tidak lagi memakai dHash
TAU_EXPLORE = 12               # batas jarak untuk eksplorasi/visualisasi pasangan
SPLIT_RATIOS = (0.70, 0.15, 0.15)   # latih / validasi / uji, berurutan menurut waktu pengambilan
BUFFER_FRAC = 0.02             # frame di tiap perbatasan subset yang dibuang (minimal 1 frame)
SEED = 42
FORCE_RESPLIT = False          # True: abaikan manifes lama dan bagi ulang

# ---------------- Sumber foto dan penyamaan gaya ----------------
PERSON_DIRS = {"steven": "./data-steven", "chela": "./data-chela", "gea": "./data-gea"}   # penanda pemilik foto
STANDARDIZE_MEMBER_PHOTOS = True   # foto kelompok di-crop ke tangan dan ditempel di kanvas putih seperti Kaggle
CANVAS_SIZE = 300              # ukuran kanvas gambar Kaggle
HAND_MARGIN = 0.13             # jarak tepi crop dari titik sendi, relatif terhadap sisi terpanjang tangan

# ---------------- Praproses dan augmentasi ----------------
IMG_SIZE = 224
ROT_DEG = 15                   # rotasi acak +-derajat
ZOOM_RANGE = (0.9, 1.15)       # faktor skala (zoom)
BRIGHTNESS = 0.3               # kecerahan acak +-30%
# Tidak ada flip horizontal (sesuai rencana).

# ---------------- Pelatihan ----------------
BATCH_SIZE = 32
NUM_WORKERS = 0 if os.name == "nt" else 2   # Windows: 0 (worker baru akan menyalin seluruh cache gambar)
MODELS_TO_RUN = ["cnn_scratch", "mobilenet_v2", "mobilenet_v3_large", "resnet50", "efficientnet_b0"]
LR_STAGE1, EPOCHS_STAGE1, PATIENCE_STAGE1 = 1e-3, 10, 3     # ekstraksi fitur
LR_STAGE2, EPOCHS_STAGE2, PATIENCE_STAGE2 = 1e-4, 20, 5     # fine-tuning (coba 1e-5 s.d. 1e-4)
LR_SCRATCH, EPOCHS_SCRATCH, PATIENCE_SCRATCH = 1e-3, 30, 6  # CNN dari nol
UNFREEZE_FRACTION = 0.5        # bagian lapisan teratas yang dibuka saat fine-tuning (1.0 = semua)
FREEZE_BN = True               # BatchNorm model pretrained tetap beku (mode inferensi)
FORCE_RETRAIN = False          # True: latih ulang walau hasil model sudah ada

# ---------------- Uji pada orang baru (bagian 13) ----------------
RUN_PERSON_HOLDOUT = True
HOLDOUT_PEOPLE = list(PERSON_DIRS)

# ---------------- Opsional: uji foto sendiri ----------------
EXTERNAL_TEST_DIR = None       # mis. "./foto_sendiri" (struktur: A/, B/, ..., Z/)

# ---------------- Mode uji cepat ----------------
if SMOKE_TEST:
    EPOCHS_STAGE1, EPOCHS_STAGE2, EPOCHS_SCRATCH = 1, 1, 2
    BENCH_RUNS = 10
    HOLDOUT_PEOPLE = HOLDOUT_PEOPLE[:1]
else:
    BENCH_RUNS = 100

OUT_DIR = Path(WORK_DIR) / ("smoke" if SMOKE_TEST else "full")
CKPT_DIR = OUT_DIR / "checkpoints"

In [ ]:
# Mount Google Drive (hanya jika diperlukan) dan siapkan folder keluaran
if IN_COLAB and (DATA_SOURCE.startswith("drive") or str(WORK_DIR).startswith("/content/drive")):
    from google.colab import drive
    drive.mount("/content/drive")

OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print("Folder keluaran:", OUT_DIR.resolve())

## 3. Memuat data
Label diambil dari nama folder (A–Z). Sel berikut mencari folder induk yang berisi 26 folder huruf, lalu memeriksa setiap gambar (ukuran, mode warna, berkas korup) sekaligus menghitung *hash* MD5 dan *perceptual hash* (dHash) untuk audit duplikat. Sumber gambar (Kaggle atau anggota) dan waktu pengambilannya dibaca dari nama berkas.

In [ ]:
LETTERS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
CLASSES = LETTERS
N_CLASSES = len(CLASSES)
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def find_class_root(base, require_all=True):
    """Cari folder yang berisi subfolder huruf A-Z. Mengembalikan (root, {huruf: Path folder})."""
    base = Path(base)
    best = None
    for d, subdirs, _ in os.walk(base):
        found = {s.upper(): Path(d) / s for s in subdirs if s.upper() in LETTERS}
        if require_all and len(found) == 26:
            return Path(d), found
        if not require_all and found and (best is None or len(found) > len(best[1])):
            best = (Path(d), found)
    if best is not None:
        return best
    raise FileNotFoundError(f"Folder huruf A-Z tidak ditemukan di {base}")


def obtain_data_base():
    if DATA_SOURCE == "kagglehub":
        if "PENGUNGGAH" in KAGGLE_SLUG:
            raise ValueError("Isi KAGGLE_SLUG dengan alamat dataset (lihat URL halaman Kaggle), "
                             "atau gunakan DATA_SOURCE lain pada sel konfigurasi.")
        try:
            import kagglehub
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
            import kagglehub
        return kagglehub.dataset_download(KAGGLE_SLUG)   # jika diminta login: kagglehub.login()
    if DATA_SOURCE == "drive_zip":
        target = Path("/content/asl_data")
        if not target.exists():
            target.mkdir(parents=True)
            with zipfile.ZipFile(DRIVE_ZIP) as z:
                z.extractall(target)
        return target
    if DATA_SOURCE == "drive_folder":
        target = Path("/content/asl_local")      # salin ke disk lokal agar pembacaan cepat
        if not target.exists():
            shutil.copytree(DRIVE_FOLDER, target)
        return target
    if DATA_SOURCE == "local":
        return Path(LOCAL_DIR)
    raise ValueError(f"DATA_SOURCE tidak dikenal: {DATA_SOURCE}")


DATA_ROOT, FOLDERS = find_class_root(obtain_data_base())
print("Folder data:", DATA_ROOT)
extra = sorted(p.name for p in DATA_ROOT.iterdir() if p.is_dir() and p.name.upper() not in LETTERS)
if extra:
    print("Folder lain (diabaikan):", extra)

rows = []
for letter in CLASSES:
    for p in sorted(FOLDERS[letter].rglob("*")):
        if p.is_file() and p.suffix.lower() in IMG_EXT:
            rows.append((str(p.relative_to(DATA_ROOT)), letter))
df_all = pd.DataFrame(rows, columns=["relpath", "label"])
df_all["label_id"] = df_all["label"].map({c: i for i, c in enumerate(CLASSES)})

# Sumber dan waktu pengambilan dari nama berkas: Image_<detik>.jpg (Kaggle), foto_<huruf>_<milidetik>.jpg (anggota)
PERSON_OF = {p.name: who for who, d in PERSON_DIRS.items()
             for p in Path(d).rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXT}


def image_source(fname):
    if fname.startswith("Image_"):
        return "kaggle"
    if fname in PERSON_OF:
        return PERSON_OF[fname]
    raise ValueError(f"Pemilik foto tidak diketahui (periksa PERSON_DIRS): {fname}")


def capture_time(fname):
    stem = Path(fname).stem
    return float(stem.split("_", 1)[1]) if stem.startswith("Image_") else float(stem.rsplit("_", 1)[1]) / 1000


names_ = df_all["relpath"].map(lambda r: Path(r).name)
df_all["source"] = names_.map(image_source)
df_all["t_capture"] = names_.map(capture_time)
print(df_all["source"].value_counts().to_string())
print("Total berkas gambar:", len(df_all))
print(df_all["label"].value_counts().sort_index().to_string())

In [ ]:
def tqdm_(it, **kw):
    try:
        from tqdm.auto import tqdm
        return tqdm(it, **kw)
    except Exception:
        return it


def dhash64(img):
    """Difference hash 64-bit: bandingkan piksel bertetangga pada citra abu-abu 9x8."""
    g = np.asarray(img.convert("L").resize((9, 8), Image.LANCZOS), dtype=np.int16)
    bits = (g[:, 1:] > g[:, :-1]).flatten()
    return f"{int(np.packbits(bits).view('>u8')[0]):016x}"   # disimpan sebagai teks hex (aman dari pembulatan float)


def scan_image(path):
    out = {"ok": False}
    try:
        with open(path, "rb") as f:
            out["md5"] = hashlib.md5(f.read()).hexdigest()
        with Image.open(path) as im:
            im.load()
            out["width"], out["height"], out["mode"] = im.size[0], im.size[1], im.mode
            out["dhash_hex"] = dhash64(im)
            out["brightness"] = float(np.asarray(im.convert("L")).mean())
        out["ok"] = True
    except Exception as e:                      # berkas korup
        out["error"] = str(e)
    return out


MANIFEST = OUT_DIR / "split_manifest.csv"
USE_MANIFEST = MANIFEST.exists() and not FORCE_RESPLIT and "source" in pd.read_csv(MANIFEST, nrows=0).columns
if USE_MANIFEST:
    print("Manifes pembagian ditemukan, dimuat ulang:", MANIFEST)
else:
    if SMOKE_TEST:   # subsampel kecil agar cepat
        def even_sample(g):
            g = g.sort_values("t_capture")
            return g.iloc[np.linspace(0, len(g) - 1, min(SMOKE_PER_SOURCE, len(g))).round().astype(int)]
        df_all = (pd.concat([even_sample(g) for _, g in df_all.groupby(["label", "source"])])
                  .sort_values(["label", "relpath"]).reset_index(drop=True))
        print("MODE UJI CEPAT: memakai", len(df_all), "gambar")
    from concurrent.futures import ThreadPoolExecutor
    paths = [DATA_ROOT / p for p in df_all["relpath"]]
    with ThreadPoolExecutor(max_workers=8) as ex:
        scans = list(tqdm_(ex.map(scan_image, paths), total=len(paths), desc="Memindai gambar"))
    scan_df = pd.DataFrame(scans)
    df_scan = pd.concat([df_all.reset_index(drop=True), scan_df], axis=1)

    bad = df_scan[~df_scan["ok"]]
    print(f"Berkas korup/tidak terbaca: {len(bad)}")
    df_scan = df_scan[df_scan["ok"]].reset_index(drop=True)
    print("Ukuran gambar (lebar x tinggi):", df_scan.groupby(["width", "height"]).size().to_dict())
    print("Mode warna:", df_scan["mode"].value_counts().to_dict())
    print("\nKecerahan rata-rata per kelas (ringkasan):")
    print(df_scan.groupby("label")["brightness"].mean().describe().round(1).to_string())

### Penyamaan gaya foto kelompok
Gambar Kaggle berupa tangan yang di-*crop* rapat di kanvas putih 300×300, sedangkan foto kelompok menampilkan seluruh pemandangan. Agar model tidak belajar membedakan sumber gambar, foto kelompok di-*crop* berdasarkan titik sendi merah yang sudah tergambar, lalu ditempel di kanvas putih dengan bingkai tipis. Proses ini dilakukan saat gambar dimuat, sehingga berkas di `data/` tidak berubah.


In [ ]:
from PIL import ImageDraw
from scipy import ndimage

STD_FALLBACK = set()   # foto yang titik merahnya tidak terbaca (memakai seluruh gambar)


def find_hand_box(rgb, border=6):
    """Kotak pembatas tangan dari titik sendi merah yang sudah tergambar. None jika titik tidak terbaca."""
    a = np.asarray(rgb).astype(np.int16)
    mask = (a[..., 0] > 170) & (a[..., 1] < 90) & (a[..., 2] < 90)
    mask[:border, :] = mask[-border:, :] = False
    mask[:, :border] = mask[:, -border:] = False
    labels, n = ndimage.label(mask)
    if n == 0:
        return None
    ids = np.arange(1, n + 1)
    sizes = ndimage.sum_labels(mask, labels, index=ids)
    keep = ids[(sizes >= 4) & (sizes <= 1000)]      # buang bintik JPEG dan benda merah besar di latar
    if len(keep) == 0 or sizes[keep - 1].sum() < 60:   # titik yang menumpuk bisa menyatu menjadi satu gumpalan
        return None
    ys, xs = np.nonzero(np.isin(labels, keep))
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())


def standardize_photo(rgb, name=None):
    """Foto kelompok -> format Kaggle: crop area tangan, bingkai tipis, di tengah kanvas putih persegi."""
    W, H = rgb.size
    edge = 5                                         # lewati bingkai tebal di tepi foto asli
    box = find_hand_box(rgb)
    if box is None:
        STD_FALLBACK.add(name)
        x0, y0, x1, y1 = edge, edge, W - 1 - edge, H - 1 - edge
    else:
        x0, y0, x1, y1 = box
        m = int(round(HAND_MARGIN * max(x1 - x0, y1 - y0)))
        x0, y0 = max(edge, x0 - m), max(edge, y0 - m)
        x1, y1 = min(W - 1 - edge, x1 + m), min(H - 1 - edge, y1 + m)
    crop = rgb.crop((x0, y0, x1 + 1, y1 + 1))
    scale = CANVAS_SIZE / max(crop.size)
    nw, nh = max(1, round(crop.size[0] * scale)), max(1, round(crop.size[1] * scale))
    crop = crop.resize((nw, nh), Image.BICUBIC)
    canvas = Image.new("RGB", (CANVAS_SIZE, CANVAS_SIZE), (255, 255, 255))
    px, py = (CANVAS_SIZE - nw) // 2, (CANVAS_SIZE - nh) // 2
    canvas.paste(crop, (px, py))
    ImageDraw.Draw(canvas).rectangle([px, py, px + nw - 1, py + nh - 1], outline=(255, 0, 255), width=3)
    return canvas


def load_image(path):
    """Baca gambar sebagai RGB. Foto kelompok (foto_*) disamakan dulu dengan format Kaggle."""
    with Image.open(path) as im:
        rgb = im.convert("RGB")
        if STANDARDIZE_MEMBER_PHOTOS and Path(path).name.startswith("foto_"):
            return standardize_photo(rgb, Path(path).name)
        return rgb.copy()


In [ ]:
# Contoh satu gambar per huruf (periksa secara visual: latar, pencahayaan, label yang janggal)
if not USE_MANIFEST:
    fig, axes = plt.subplots(2, 13, figsize=(16, 3.6))
    rng = np.random.RandomState(SEED)
    for ax, c in zip(axes.ravel(), CLASSES):
        sub = df_scan[df_scan["label"] == c]
        p = DATA_ROOT / sub.iloc[rng.randint(len(sub))]["relpath"]
        ax.imshow(load_image(p)); ax.set_title(c); ax.axis("off")
    plt.suptitle("Satu contoh acak per huruf"); plt.tight_layout(); plt.show()

## 4. Audit duplikat dan gambar nyaris kembar
Urutan: (a) duplikat persis (MD5), (b) pasangan nyaris kembar dalam kelas yang sama (jarak Hamming dHash ≤ ambang), (c) pasangan nyaris kembar **lintas kelas** (indikasi label salah). Bagian ini hanya untuk laporan audit: setiap huruf pada data Kaggle berupa rangkaian frame beruntun (sekitar 14 detik), sehingga kelompok dHash saling menyambung dan tidak dapat dipakai untuk membagi data. Pembagian di bagian 5 memakai urutan waktu.

In [ ]:
POP8 = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def hamming_block(h_rows, h_all):
    """Jarak Hamming antara setiap elemen h_rows dan seluruh h_all (uint64). Hasil: matriks (r, N)."""
    x = np.bitwise_xor(h_rows[:, None], h_all[None, :])
    return POP8[x.view(np.uint8).reshape(x.shape[0], x.shape[1], 8)].sum(axis=2).astype(np.int16)


if not USE_MANIFEST:
    # (a) duplikat persis
    n0 = len(df_scan)
    nlab = df_scan.groupby("md5")["label"].nunique()
    conflict_md5 = set(nlab[nlab > 1].index)
    n_conflict = int(df_scan["md5"].isin(conflict_md5).sum())
    if conflict_md5:
        print(f"PERINGATAN: {len(conflict_md5)} gambar identik muncul di lebih dari satu kelas "
              f"({n_conflict} berkas). Semua salinannya dibuang karena labelnya ambigu.")
        df_scan = df_scan[~df_scan["md5"].isin(conflict_md5)]
    n_dup = int(df_scan.duplicated("md5").sum())
    print(f"Duplikat persis (label sama): {n_dup} berkas berlebih")
    if DROP_EXACT_DUPLICATES:
        df_scan = df_scan.drop_duplicates("md5", keep="first")
    df = df_scan.reset_index(drop=True)
    print(f"Jumlah gambar: {n0} -> {len(df)}")

    # (b) dan (c) analisis tetangga terdekat berdasarkan dHash
    h = np.array([int(x, 16) for x in df["dhash_hex"]], dtype=np.uint64)
    lab = df["label_id"].to_numpy()
    n = len(df)
    nn_within = np.full(n, 65, np.int16)
    nn_cross = np.full(n, 65, np.int16)
    pw, pc = [], []        # pasangan (i, j, jarak) dalam kelas / lintas kelas
    CH = 256
    for s in tqdm_(range(0, n, CH), desc="Menghitung jarak hash"):
        e = min(n, s + CH)
        d = hamming_block(h[s:e], h)
        d[np.arange(e - s), np.arange(s, e)] = 65               # abaikan diri sendiri
        same = lab[s:e, None] == lab[None, :]
        nn_within[s:e] = np.where(same, d, 65).min(axis=1)
        nn_cross[s:e] = np.where(~same, d, 65).min(axis=1)
        ii, jj = np.nonzero(d <= TAU_EXPLORE)
        gi = ii + s
        keep = gi < jj
        for a, b, dist, sm in zip(gi[keep], jj[keep], d[ii[keep], jj[keep]], same[ii[keep], jj[keep]]):
            (pw if sm else pc).append((int(a), int(b), int(dist)))
    pairs_within = pd.DataFrame(pw, columns=["i", "j", "d"])
    pairs_cross = pd.DataFrame(pc, columns=["i", "j", "d"])
    df["nn_within"], df["nn_cross"] = nn_within, nn_cross

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    ax[0].hist(np.clip(nn_within, 0, 20), bins=np.arange(-0.5, 21.5, 1), color="#4C72B0")
    ax[0].axvline(NEAR_DUP_THRESHOLD + 0.5, color="red", ls="--", label=f"ambang = {NEAR_DUP_THRESHOLD}")
    ax[0].set_title("Jarak ke tetangga terdekat (kelas sama)"); ax[0].set_xlabel("jarak Hamming (>=20 digabung)")
    ax[0].set_ylabel("jumlah gambar"); ax[0].legend()
    ax[1].hist(np.clip(nn_cross, 0, 20), bins=np.arange(-0.5, 21.5, 1), color="#DD8452")
    ax[1].axvline(NEAR_DUP_THRESHOLD + 0.5, color="red", ls="--")
    ax[1].set_title("Jarak ke tetangga terdekat (kelas lain)"); ax[1].set_xlabel("jarak Hamming (>=20 digabung)")
    plt.tight_layout(); plt.show()

    print(f"Pasangan nyaris kembar (d <= {NEAR_DUP_THRESHOLD}) dalam kelas sama : "
          f"{int((pairs_within['d'] <= NEAR_DUP_THRESHOLD).sum())}")
    print(f"Pasangan nyaris kembar (d <= {NEAR_DUP_THRESHOLD}) LINTAS kelas      : "
          f"{int((pairs_cross['d'] <= NEAR_DUP_THRESHOLD).sum())}  (periksa manual jika > 0)")

In [ ]:
# Contoh pasangan menurut jaraknya, untuk menilai ambang yang pantas.
# Kolom kiri = jarak kecil (hampir identik). Tentukan jarak terbesar yang masih terlihat seperti "foto yang sama".
def show_pairs(pairs, title, dists=(0, 2, 4, 6, 8, 10, 12)):
    chosen = []
    for dd in dists:
        sub = pairs[pairs["d"] == dd]
        if len(sub) == 0:
            sub = pairs[pairs["d"] == dd + 1]
        if len(sub):
            chosen.append(sub.iloc[len(sub) // 2])
    if not chosen:
        print(title, ": tidak ada pasangan pada rentang jarak ini."); return
    fig, axes = plt.subplots(2, len(chosen), figsize=(2.2 * len(chosen), 4.8), squeeze=False)
    for k, r in enumerate(chosen):
        for row, idx in enumerate((int(r["i"]), int(r["j"]))):
            axes[row, k].imshow(load_image(DATA_ROOT / df.loc[idx, "relpath"]))
            axes[row, k].axis("off")
        axes[0, k].set_title(f"d={int(r['d'])}  ({df.loc[int(r['i']), 'label']})")
    plt.suptitle(title); plt.tight_layout(); plt.show()


if not USE_MANIFEST:
    show_pairs(pairs_within, "Pasangan nyaris kembar dalam kelas yang sama")
    if len(pairs_cross):
        show_pairs(pairs_cross, "Pasangan serupa LINTAS kelas (kemungkinan label salah atau huruf sangat mirip)")

## 5. Pembagian data 70/15/15 berdasarkan urutan waktu
Gambar Kaggle berupa rangkaian frame beruntun, sehingga pembagian acak menempatkan frame yang hampir identik di data latih dan data uji. Karena itu, gambar setiap pasangan sumber dan huruf diurutkan menurut waktu pengambilan, lalu dibagi menjadi blok berurutan: latih, validasi, uji. Beberapa frame di setiap perbatasan dibuang (`split = "buffer"`). Setelah dibagi, dilakukan verifikasi otomatis.

Hasil data uji pada pembagian ini menggambarkan kinerja pada orang dan sesi yang sudah dikenal model. Kinerja pada orang baru diukur terpisah di bagian 13.


In [ ]:
def block_split(frame, ratios, buffer_frac):
    """Per (sumber, huruf): urutkan menurut waktu, bagi jadi blok latih-val-uji, buang frame di perbatasan."""
    out = pd.Series("", index=frame.index, dtype=object)
    for key, g in frame.groupby(["source", "label_id"]):
        idx = g.sort_values("t_capture").index.to_numpy()
        n = len(idx)
        b = max(1, int(round(buffer_frac * n)))
        usable = n - 2 * b
        if usable < 3:
            raise ValueError(f"Gambar terlalu sedikit untuk dibagi: {key}, n={n}")
        n_tr = int(round(ratios[0] * usable))
        n_va = max(1, int(round(ratios[1] * usable)))
        n_te = usable - n_tr - n_va
        out.loc[idx] = ["train"] * n_tr + ["buffer"] * b + ["val"] * n_va + ["buffer"] * b + ["test"] * n_te
    return out


if USE_MANIFEST:
    df = pd.read_csv(MANIFEST)
    print("Manifes dimuat:", len(df), "gambar")
else:
    df["split"] = block_split(df, SPLIT_RATIOS, BUFFER_FRAC)
    keep_cols = ["relpath", "label", "label_id", "source", "t_capture", "md5", "dhash_hex", "split", "brightness"]
    out = df[keep_cols].copy()
    out.to_csv(MANIFEST, index=False)
    print("Manifes disimpan:", MANIFEST)
    df = out

print("\nJumlah gambar per subset:")
print(pd.crosstab(df["label"], df["split"]).assign(Total=lambda t: t.sum(axis=1)).T.to_string())
print("\nPer sumber:")
print(pd.crosstab(df["source"], df["split"], margins=True).to_string())
print("\nTotal:", df["split"].value_counts().to_dict())


In [ ]:
# Verifikasi pembagian
used = df[df["split"] != "buffer"]
assert (df.groupby("md5")["split"].nunique() == 1).all(), "Ada gambar identik di lebih dari satu subset!"
assert set(used["split"]) == {"train", "val", "test"}, "Ada subset yang kosong!"
gaps = []
for key, g in used.groupby(["source", "label_id"]):
    t = g.groupby("split")["t_capture"].agg(["min", "max"])
    assert t.loc["train", "max"] < t.loc["val", "min"] and t.loc["val", "max"] < t.loc["test", "min"], f"Blok waktu bertumpuk: {key}"
    gaps.append({"source": key[0], "jeda latih-val (detik)": t.loc["val", "min"] - t.loc["train", "max"],
                 "jeda val-uji (detik)": t.loc["test", "min"] - t.loc["val", "max"]})
print("OK: tidak ada gambar identik lintas subset, dan blok waktu latih < validasi < uji pada setiap sumber dan huruf.")
print("\nMedian jeda waktu antar-blok per sumber:")
display(pd.DataFrame(gaps).groupby("source").median().round(2))

h_all = np.array([int(x, 16) for x in df["dhash_hex"]], dtype=np.uint64)
rows_v = []
for sp in ["val", "test"]:
    nn = []
    for cid in range(N_CLASSES):
        tr = np.where((df["split"] == "train") & (df["label_id"] == cid))[0]
        te = np.where((df["split"] == sp) & (df["label_id"] == cid))[0]
        if len(tr) and len(te):
            nn.append(hamming_block(h_all[te], h_all[tr]).min(axis=1))
    nn = np.concatenate(nn)
    rows_v.append({"subset": sp, "jarak minimum ke data latih": int(nn.min()),
                   f"% gambar dengan jarak <= {NEAR_DUP_THRESHOLD}": round(100 * float((nn <= NEAR_DUP_THRESHOLD).mean()), 2)})
display(pd.DataFrame(rows_v))
print("Catatan: gambar uji tetap berasal dari sesi dan orang yang sama dengan data latih, sehingga masih mirip. "
      "Ukuran kinerja pada orang baru ada di bagian 13.")

df_train = df[df["split"] == "train"].reset_index(drop=True)
df_val = df[df["split"] == "val"].reset_index(drop=True)
df_test = df[df["split"] == "test"].reset_index(drop=True)
print(len(df_train), len(df_val), len(df_test))


## 6. Dataset PyTorch, praproses, dan augmentasi
Normalisasi memakai rata-rata dan simpangan baku ImageNet (sesuai bobot *pretrained* torchvision). Augmentasi hanya untuk data latih: rotasi kecil, *zoom*, dan kecerahan. **Tidak ada flip horizontal.**

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models
from torchvision import transforms as T
from sklearn.metrics import (confusion_matrix, classification_report,
                             precision_recall_fscore_support)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| device:", device,
      "|", torch.cuda.get_device_name(0) if USE_AMP else "tanpa GPU (pelatihan akan sangat lambat)")


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
torch.backends.cudnn.benchmark = True    # lebih cepat; hasil tidak identik bit-per-bit antar-sesi

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomAffine(degrees=ROT_DEG, scale=ZOOM_RANGE, fill=(255, 255, 255)),
    T.ColorJitter(brightness=BRIGHTNESS),
    T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])


IMAGE_CACHE = {}   # dipakai bersama semua loader; gambar disimpan sudah berukuran IMG_SIZE


def cached_image(path):
    key = str(path)
    if key not in IMAGE_CACHE:
        IMAGE_CACHE[key] = load_image(path).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    return IMAGE_CACHE[key]


class ASLDataset(Dataset):
    def __init__(self, frame, root, transform, cache=True):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
        self.labels = self.frame["label_id"].to_numpy()
        self.paths = [Path(root) / p for p in self.frame["relpath"]]
        self.images = [cached_image(p) for p in self.paths] if cache else None   # cache di RAM (+- 1,5 GB)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, i):
        img = self.images[i] if self.images is not None else load_image(self.paths[i])
        return self.transform(img), int(self.labels[i])


def make_loader(frame, root, transform, shuffle):
    ds = ASLDataset(frame, root, transform)
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=USE_AMP, persistent_workers=NUM_WORKERS > 0, generator=g)


train_loader = make_loader(df_train, DATA_ROOT, train_tf, True)
val_loader = make_loader(df_val, DATA_ROOT, eval_tf, False)
test_loader = make_loader(df_test, DATA_ROOT, eval_tf, False)
print("Batch per epoch (latih/val/uji):", len(train_loader), len(val_loader), len(test_loader))

In [ ]:
# Pratinjau augmentasi: satu gambar latih yang sama, delapan kali diaugmentasi
def denorm(x):
    m = torch.tensor(MEAN).view(3, 1, 1); s = torch.tensor(STD).view(3, 1, 1)
    return (x * s + m).clamp(0, 1).permute(1, 2, 0).numpy()


base_img = train_loader.dataset.images[0]
fig, axes = plt.subplots(1, 9, figsize=(18, 2.4))
axes[0].imshow(base_img.resize((IMG_SIZE, IMG_SIZE))); axes[0].set_title("asli"); axes[0].axis("off")
for ax in axes[1:]:
    ax.imshow(denorm(train_tf(base_img))); ax.axis("off")
plt.suptitle("Pratinjau augmentasi (tanpa flip)"); plt.tight_layout(); plt.show()

## 7. Model
Empat model *pretrained* ImageNet (lapisan klasifikasi akhir diganti menjadi 26 kelas) dan satu CNN sederhana yang dilatih dari bobot acak. "Kepala" yang dilatih pada tahap 1 adalah `classifier` (MobileNet, EfficientNet) atau `fc` (ResNet50). Pada MobileNetV3, kepala ini juga memuat satu lapisan tersembunyi bawaannya.

In [ ]:
DISPLAY = {"cnn_scratch": "CNN dari nol", "mobilenet_v2": "MobileNetV2", "mobilenet_v3_large": "MobileNetV3-Large",
           "resnet50": "ResNet50", "efficientnet_b0": "EfficientNet-B0"}
PRETRAINED = ["mobilenet_v2", "mobilenet_v3_large", "resnet50", "efficientnet_b0"]


class ScratchCNN(nn.Module):
    """CNN sederhana: 4 blok (2 x [Conv3x3-BN-ReLU] + MaxPool), GAP, dropout, linear."""
    def __init__(self, num_classes, widths=(32, 64, 128, 256), p_drop=0.3):
        super().__init__()
        layers, c_in = [], 3
        for w in widths:
            layers += [nn.Conv2d(c_in, w, 3, padding=1, bias=False), nn.BatchNorm2d(w), nn.ReLU(inplace=True),
                       nn.Conv2d(w, w, 3, padding=1, bias=False), nn.BatchNorm2d(w), nn.ReLU(inplace=True),
                       nn.MaxPool2d(2)]
            c_in = w
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(p_drop), nn.Linear(c_in, num_classes))

    def forward(self, x):
        return self.head(self.features(x))


def build_model(name, num_classes, pretrained=True):
    if name == "cnn_scratch":
        return ScratchCNN(num_classes)
    m = models.get_model(name, weights="IMAGENET1K_V1" if pretrained else None)
    if name == "resnet50":
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    else:   # mobilenet_v2, mobilenet_v3_large, efficientnet_b0: lapisan terakhir classifier
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
    return m


def get_head(model, name):
    if name == "cnn_scratch":
        return model.head
    return model.fc if name == "resnet50" else model.classifier


def count_params(model):
    return sum(p.numel() for p in model.parameters())


BN_TYPES = nn.modules.batchnorm._BatchNorm


def set_trainable(model, name, stage):
    """stage: 'head' (ekstraksi fitur), 'finetune' (kepala + UNFREEZE_FRACTION lapisan teratas), 'all'."""
    for p in model.parameters():
        p.requires_grad = (stage == "all")
    if stage == "all":
        return
    for p in get_head(model, name).parameters():
        p.requires_grad = True
    if stage == "finetune":
        leaves = [m for m in model.modules()
                  if len(list(m.children())) == 0 and len(list(m.parameters(recurse=False))) > 0]
        k = int(round(len(leaves) * UNFREEZE_FRACTION))
        for m in leaves[len(leaves) - k:]:
            if FREEZE_BN and isinstance(m, BN_TYPES):
                continue
            for p in m.parameters():
                p.requires_grad = True


def set_train_mode(model, name, stage):
    if stage == "head":                  # backbone beku: mode inferensi penuh, hanya kepala yang 'train'
        model.eval()
        get_head(model, name).train()
    else:
        model.train()
        if FREEZE_BN and name != "cnn_scratch":
            for m in model.modules():
                if isinstance(m, BN_TYPES):
                    m.eval()


# Ringkasan model (bobot acak agar cepat; jumlah parameter sama dengan versi pretrained)
rows_m = []
for n_ in MODELS_TO_RUN:
    mm = build_model(n_, N_CLASSES, pretrained=False)
    rows_m.append({"Model": DISPLAY[n_], "Parameter (juta)": round(count_params(mm) / 1e6, 2)})
    del mm
display(pd.DataFrame(rows_m))

## 8. Fungsi pelatihan dan evaluasi
Pelatihan memakai Adam, *cross-entropy*, *mixed precision* (jika GPU), *early stopping* pada *validation loss* yang dirata-ratakan per sumber gambar (agar tidak didominasi Kaggle), dan menyimpan *checkpoint* terbaik. Tahap 2 dimulai dari *checkpoint* terbaik tahap 1 dan hanya dipakai jika *validation loss*-nya lebih baik.

In [ ]:
def train_epoch(model, name, stage, loader, optimizer, scaler, criterion):
    set_train_mode(model, name, stage)
    tot_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        tot_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        n += len(y)
    return tot_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion=None):
    model.eval()
    tot_loss, preds, targets = 0.0, [], []
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x)
            if criterion is not None:
                tot_loss += criterion(logits.float(), y).item() * len(y)
        preds.append(logits.argmax(1).cpu()); targets.append(y.cpu())
    preds, targets = torch.cat(preds).numpy(), torch.cat(targets).numpy()
    return tot_loss / len(targets), float((preds == targets).mean()), preds, targets


@torch.no_grad()
def evaluate_balanced(model, loader):
    """Loss dan akurasi dihitung per sumber gambar lalu dirata-ratakan, sehingga setiap sumber berbobot sama."""
    model.eval()
    losses, hits = [], []
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x)
        losses.append(nn.functional.cross_entropy(logits.float(), y, reduction="none").cpu())
        hits.append((logits.argmax(1) == y).cpu())
    losses, hits = torch.cat(losses).numpy(), torch.cat(hits).numpy().astype(float)
    sources = loader.dataset.frame["source"].to_numpy()
    groups = [sources == s for s in np.unique(sources)]
    return float(np.mean([losses[g].mean() for g in groups])), float(np.mean([hits[g].mean() for g in groups]))


def fit_stage(model, name, stage, tag, lr, max_epochs, patience, ckpt_path, loaders, epoch_offset=0, start_best=None):
    """start_best: baris riwayat terbaik tahap sebelumnya; tahap ini harus mengalahkannya agar checkpoint berubah."""
    tr_loader, va_loader = loaders
    set_trainable(model, name, stage)
    params = [p for p in model.parameters() if p.requires_grad]
    print(f"[{DISPLAY[name]} | {tag}] parameter yang dilatih: {sum(p.numel() for p in params) / 1e6:.2f} juta, lr={lr}")
    optimizer = torch.optim.Adam(params, lr=lr)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    criterion = nn.CrossEntropyLoss()
    best, bad, hist, best_row = (start_best["val_loss"] if start_best else float("inf")), 0, [], start_best
    for ep in range(1, max_epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_epoch(model, name, stage, tr_loader, optimizer, scaler, criterion)
        va_loss, va_acc = evaluate_balanced(model, va_loader)
        row = dict(stage=tag, epoch=ep, global_epoch=epoch_offset + ep, train_loss=tr_loss, train_acc=tr_acc,
                   val_loss=va_loss, val_acc=va_acc, seconds=time.time() - t0)
        hist.append(row)
        flag = ""
        if va_loss < best - 1e-4:
            best, bad, best_row = va_loss, 0, row
            torch.save(model.state_dict(), ckpt_path)
            flag = " *"
        else:
            bad += 1
        print(f"  ep {ep:02d}  loss {tr_loss:.4f}/{va_loss:.4f}  acc {tr_acc:.4f}/{va_acc:.4f}  ({row['seconds']:.0f}s){flag}")
        if bad >= patience:
            print(f"  early stopping (tidak membaik {patience} epoch)")
            break
    model.load_state_dict(torch.load(ckpt_path, map_location=device))   # pulihkan bobot terbaik
    return hist, best_row


def bootstrap_ci(correct, n_boot=2000, seed=0):
    rng = np.random.RandomState(seed)
    idx = rng.randint(0, len(correct), size=(n_boot, len(correct)))
    accs = correct[idx].mean(axis=1)
    return float(np.percentile(accs, 2.5)), float(np.percentile(accs, 97.5))


@torch.no_grad()
def benchmark_latency(model, dev_name, n_warm=10, n_run=100):
    """Latensi inferensi 1 gambar (ms), fp32, tanpa augmentasi. Mengembalikan (rata-rata, median)."""
    dev = torch.device(dev_name)
    m = copy.deepcopy(model).to(dev).eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=dev)
    for _ in range(n_warm):
        m(x)
    ts = []
    for _ in range(n_run):
        if dev.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        m(x)
        if dev.type == "cuda":
            torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000)
    del m
    if dev.type == "cuda":
        torch.cuda.empty_cache()
    return float(np.mean(ts)), float(np.median(ts))

## 9. Menjalankan eksperimen
Setiap model dilatih, dievaluasi pada data uji, diukur ukuran dan kecepatannya, lalu hasilnya disimpan (`result_<model>.json`, riwayat pelatihan, dan prediksi uji). Model yang hasilnya sudah ada akan dilewati kecuali `FORCE_RETRAIN = True`.

Data uji **tidak** dipakai untuk memilih *checkpoint*, *early stopping*, maupun hiperparameter (semuanya memakai data validasi). Untuk model *pretrained*, hasil uji setelah tahap 1 juga dicatat hanya sebagai pelaporan untuk menjawab rumusan masalah 2.

In [ ]:
def macro_metrics(y, p):
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, average="macro", zero_division=0)
    return float(pr), float(rc), float(f1)


def run_experiment(name, out_dir=OUT_DIR, loaders=None):
    """Latih dan evaluasi satu model. loaders = (latih, validasi, uji); bawaan: pembagian bagian 5."""
    tr_loader, va_loader, te_loader = loaders or (train_loader, val_loader, test_loader)
    out_dir, ckpt_dir = Path(out_dir), Path(out_dir) / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    res_path = out_dir / f"result_{name}.json"
    if res_path.exists() and not FORCE_RETRAIN:
        print(f"Lewati {DISPLAY[name]}: hasil sudah ada ({res_path.name})")
        return json.loads(res_path.read_text())
    print("=" * 70, f"\n{DISPLAY[name]}\n" + "=" * 70)
    seed_everything(SEED)
    tr_loader.generator.manual_seed(SEED)   # urutan batch sama untuk setiap model, juga saat run dilanjutkan
    t_start = time.time()
    pretrained = name != "cnn_scratch"
    model = build_model(name, N_CLASSES, pretrained=pretrained).to(device)
    final_ckpt = ckpt_dir / f"{name}_best.pt"
    res = {"name": name, "display": DISPLAY[name]}

    if pretrained:
        h1, b1 = fit_stage(model, name, "head", "tahap1", LR_STAGE1, EPOCHS_STAGE1, PATIENCE_STAGE1,
                           ckpt_dir / f"{name}_stage1.pt", (tr_loader, va_loader))
        _, test_acc1, p1, y1 = evaluate(model, te_loader)
        res.update(val_acc_stage1=b1["val_acc"], test_acc_stage1=test_acc1,
                   test_f1_stage1=macro_metrics(y1, p1)[2], epochs_stage1=len(h1))
        shutil.copy(ckpt_dir / f"{name}_stage1.pt", final_ckpt)   # jika tahap 2 tidak membaik, bobot tahap 1 yang dipakai
        h2, b2 = fit_stage(model, name, "finetune", "tahap2", LR_STAGE2, EPOCHS_STAGE2, PATIENCE_STAGE2,
                           final_ckpt, (tr_loader, va_loader), epoch_offset=len(h1), start_best=b1)
        hist, best = h1 + h2, b2
        res["epochs_stage2"] = len(h2)
        res["final_stage"] = b2["stage"]
    else:
        hist, best = fit_stage(model, name, "all", "scratch", LR_SCRATCH, EPOCHS_SCRATCH, PATIENCE_SCRATCH, final_ckpt,
                               (tr_loader, va_loader))
        res["epochs_scratch"] = len(hist)
    train_minutes = (time.time() - t_start) / 60

    # Evaluasi akhir pada data uji
    _, test_acc, preds, targets = evaluate(model, te_loader)
    correct = (preds == targets).astype(float)
    sources = te_loader.dataset.frame["source"].to_numpy()
    res["test_acc_by_source"] = {s: float(correct[sources == s].mean()) for s in np.unique(sources)}
    lo, hi = bootstrap_ci(correct)
    pr, rc, f1 = macro_metrics(targets, preds)
    np.savez(out_dir / f"test_pred_{name}.npz", y=targets, p=preds)
    pd.DataFrame(hist).to_csv(out_dir / f"history_{name}.csv", index=False)

    # Ukuran dan kecepatan
    lat_gpu = benchmark_latency(model, "cuda", n_run=BENCH_RUNS) if USE_AMP else (float("nan"), float("nan"))
    lat_cpu = benchmark_latency(model, "cpu", n_run=BENCH_RUNS)
    res.update(
        params_million=count_params(model) / 1e6,
        size_mb=os.path.getsize(final_ckpt) / 1e6,
        best_val_loss=best["val_loss"], best_val_acc=best["val_acc"],
        test_acc=test_acc, test_acc_ci_low=lo, test_acc_ci_high=hi,
        test_precision_macro=pr, test_recall_macro=rc, test_f1_macro=f1,
        latency_gpu_ms_mean=lat_gpu[0], latency_gpu_ms_median=lat_gpu[1],
        latency_cpu_ms_mean=lat_cpu[0], latency_cpu_ms_median=lat_cpu[1],
        train_minutes=train_minutes, torch_threads=torch.get_num_threads())
    res_path.write_text(json.dumps(res, indent=2))
    print(f"\n>> {DISPLAY[name]}: akurasi uji {test_acc:.4f} (CI95 {lo:.4f}-{hi:.4f}), macro-F1 {f1:.4f}, "
          f"{res['params_million']:.2f} juta parameter, {res['size_mb']:.1f} MB, "
          f"CPU {lat_cpu[0]:.1f} ms, waktu latih {train_minutes:.1f} menit")
    del model
    if USE_AMP:
        torch.cuda.empty_cache()
    return res


RESULTS = {}
for name in MODELS_TO_RUN:
    RESULTS[name] = run_experiment(name)

## 10. Kurva pelatihan
Garis putus-putus menandai pergantian dari tahap 1 (ekstraksi fitur) ke tahap 2 (*fine-tuning*).

In [ ]:
def plot_history(name):
    h = pd.read_csv(OUT_DIR / f"history_{name}.csv")
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    for a, (c1, c2, ttl) in zip(ax, [("train_loss", "val_loss", "Loss"), ("train_acc", "val_acc", "Akurasi")]):
        a.plot(h["global_epoch"], h[c1], marker="o", ms=3, label="latih")
        a.plot(h["global_epoch"], h[c2], marker="o", ms=3, label="validasi")
        starts = h.groupby("stage")["global_epoch"].min().sort_values().iloc[1:]
        for b in starts:
            a.axvline(b - 0.5, color="gray", ls="--")
        a.set_title(f"{DISPLAY[name]}: {ttl}"); a.set_xlabel("epoch"); a.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"curve_{name}.png", dpi=150)
    plt.show()


for name in MODELS_TO_RUN:
    plot_history(name)

## 11. Evaluasi pada frame lanjutan dari sesi yang sudah dikenal dan analisis kesalahan
Menampilkan laporan klasifikasi per huruf, *confusion matrix*, pasangan huruf yang paling sering tertukar, kinerja J dan Z, serta contoh gambar yang salah diklasifikasikan.

In [ ]:
def load_pred(name):
    d = np.load(OUT_DIR / f"test_pred_{name}.npz")
    return d["y"], d["p"]


def per_class_table(name):
    y, p = load_pred(name)
    pr, rc, f1, sup = precision_recall_fscore_support(y, p, labels=list(range(N_CLASSES)), zero_division=0)
    return pd.DataFrame({"Huruf": CLASSES, "Precision": pr, "Recall": rc, "F1": f1, "Jumlah uji": sup})


def plot_confusion(name):
    y, p = load_pred(name)
    cm = confusion_matrix(y, p, labels=list(range(N_CLASSES)))
    cmn = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None)
    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CLASSES)
    ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASSES)
    ax.set_xlabel("Prediksi"); ax.set_ylabel("Label sebenarnya")
    ax.set_title(f"Confusion matrix (dinormalisasi per baris): {DISPLAY[name]}")
    fig.colorbar(im, fraction=0.046)
    plt.tight_layout(); plt.savefig(OUT_DIR / f"confusion_{name}.png", dpi=150); plt.show()
    return cm


def top_confusions(cm, k=10):
    off = cm.copy().astype(float)
    np.fill_diagonal(off, 0)
    order = np.dstack(np.unravel_index(np.argsort(-off, axis=None), off.shape))[0][:k]
    rows = [{"Benar": CLASSES[i], "Diprediksi": CLASSES[j], "Jumlah": int(cm[i, j]),
             "% dari kelas benar": round(100 * cm[i, j] / max(cm[i].sum(), 1), 1)} for i, j in order if cm[i, j] > 0]
    return pd.DataFrame(rows)


def show_errors(name, max_n=24):
    y, p = load_pred(name)
    wrong = np.where(y != p)[0]
    print(f"{DISPLAY[name]}: {len(wrong)} kesalahan dari {len(y)} gambar uji")
    if len(wrong) == 0:
        return
    sel = wrong[:max_n]
    cols = 8
    rws = int(np.ceil(len(sel) / cols))
    fig, axes = plt.subplots(rws, cols, figsize=(2 * cols, 2.3 * rws), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, i in zip(axes.ravel(), sel):
        ax.imshow(load_image(DATA_ROOT / df_test.loc[i, "relpath"]))
        ax.set_title(f"benar {CLASSES[y[i]]} / pred {CLASSES[p[i]]}", fontsize=9)
    plt.tight_layout(); plt.show()


# Hanya untuk memilih contoh analisis kesalahan: validasi tidak memprediksi kinerja pada orang baru (bagian 14)
BEST = max(RESULTS, key=lambda n: RESULTS[n]["best_val_acc"])
print("Model dengan validasi tertinggi (bukan berarti terbaik untuk orang baru):", DISPLAY[BEST])

for name in MODELS_TO_RUN:
    print("\n" + "=" * 70, f"\n{DISPLAY[name]}")
    pc = per_class_table(name)
    display(pc.round(3))
    print("Kinerja huruf J dan Z (aslinya bergerak):")
    display(pc[pc["Huruf"].isin(["J", "Z"])].round(3))
    cm = plot_confusion(name)
    print("Pasangan huruf yang paling sering tertukar:")
    display(top_confusions(cm))
    show_errors(name)

## 12. Perbandingan model pada sesi yang sudah dikenal
Data uji di sini adalah **frame lanjutan dari rekaman yang juga ada di data latih**, jadi angkanya bukan ukuran kinerja pada pengguna baru. Ukuran generalisasi utama ada di bagian 13 dan 14.

Akurasi validasi dihitung sebagai rata-rata per sumber, maka bandingkan dengan kolom *rata-rata per sumber*, bukan kolom *per gambar* yang didominasi Kaggle.

In [ ]:
rows_c = []
for name in MODELS_TO_RUN:
    r = RESULTS[name]
    rows_c.append({
        "Model": r["display"],
        "Akurasi sesi dikenal (per gambar)": round(r["test_acc"], 4),
        "Akurasi sesi dikenal (rata-rata per sumber)": round(float(np.mean(list(r["test_acc_by_source"].values()))), 4),
        "CI 95% per frame (terlalu sempit, lihat bagian 14)": f"{r['test_acc_ci_low']:.4f}-{r['test_acc_ci_high']:.4f}",
        "Precision (macro)": round(r["test_precision_macro"], 4),
        "Recall (macro)": round(r["test_recall_macro"], 4),
        "F1 (macro)": round(r["test_f1_macro"], 4),
        "Akurasi uji setelah tahap 1": round(r["test_acc_stage1"], 4) if "test_acc_stage1" in r else None,
        "Akurasi validasi (rata-rata per sumber)": round(r["best_val_acc"], 4),
        "Parameter (juta)": round(r["params_million"], 2),
        "Ukuran (MB)": round(r["size_mb"], 1),
        "Latensi GPU (ms)": round(r["latency_gpu_ms_median"], 2),
        "Latensi CPU (ms)": round(r["latency_cpu_ms_median"], 1),
        "Waktu latih (menit)": round(r["train_minutes"], 1)})
comparison = pd.DataFrame(rows_c)
comparison.to_csv(OUT_DIR / "comparison.csv", index=False)
display(comparison)

by_source = pd.DataFrame({RESULTS[n]["display"]: RESULTS[n]["test_acc_by_source"] for n in MODELS_TO_RUN}).T
by_source["Rata-rata per sumber"] = by_source.mean(axis=1)
by_source = by_source.round(4)
by_source.to_csv(OUT_DIR / "comparison_by_source.csv")
print("Akurasi pada sesi yang sudah dikenal, per sumber gambar:")
display(by_source)
print("Latensi = median 1 gambar (batch 1, fp32). Jumlah thread CPU:", RESULTS[MODELS_TO_RUN[0]]["torch_threads"],
      "| Angka bergantung pada perangkat (RTX 2060 dan CPU laptop); gunakan sebagai perbandingan relatif.")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for n_, r in RESULTS.items():
    ax[0].scatter(r["size_mb"], r["test_acc"] * 100, s=60)
    ax[0].annotate(r["display"], (r["size_mb"], r["test_acc"] * 100), textcoords="offset points", xytext=(5, 5), fontsize=9)
    ax[1].scatter(r["latency_cpu_ms_median"], r["test_acc"] * 100, s=60)
    ax[1].annotate(r["display"], (r["latency_cpu_ms_median"], r["test_acc"] * 100), textcoords="offset points", xytext=(5, 5), fontsize=9)
ax[0].set_xlabel("Ukuran berkas bobot (MB)"); ax[0].set_ylabel("Akurasi sesi dikenal (%)"); ax[0].set_title("Akurasi sesi dikenal vs ukuran model")
ax[1].set_xlabel("Latensi CPU (ms/gambar)"); ax[1].set_ylabel("Akurasi sesi dikenal (%)"); ax[1].set_title("Akurasi sesi dikenal vs kecepatan inferensi CPU")
plt.tight_layout(); plt.savefig(OUT_DIR / "comparison.png", dpi=150); plt.show()

## 13. Uji pada orang baru (ukuran generalisasi utama)
Foto setiap anggota ditahan bergantian: model dilatih dan divalidasi dengan Kaggle serta foto dua anggota lainnya (blok latih dan validasi dari bagian 5), lalu diuji pada **seluruh** foto anggota yang ditahan. Hasilnya menggambarkan kinerja pada orang yang belum pernah dilihat model. Setiap model dilatih ulang untuk setiap orang, jadi bagian ini memakan waktu sekitar tiga kali pelatihan bagian 9. Hasil disimpan di `holdout/<nama>/`, dan model yang sudah selesai dilewati jika dijalankan ulang.


In [ ]:
def holdout_loaders(person):
    """Latih dan validasi tanpa foto orang ini; seluruh fotonya menjadi data uji."""
    others = df[df["source"] != person]
    return (make_loader(others[others["split"] == "train"], DATA_ROOT, train_tf, True),
            make_loader(others[others["split"] == "val"], DATA_ROOT, eval_tf, False),
            make_loader(df[df["source"] == person], DATA_ROOT, eval_tf, False))


HOLDOUT_RESULTS = {}
if RUN_PERSON_HOLDOUT:
    for person in HOLDOUT_PEOPLE:
        print("#" * 70, f"\nORANG YANG DITAHAN: {person}\n" + "#" * 70)
        loaders = holdout_loaders(person)
        assert person not in set(loaders[0].dataset.frame["source"]) | set(loaders[1].dataset.frame["source"])
        for name in MODELS_TO_RUN:
            HOLDOUT_RESULTS[(person, name)] = run_experiment(name, OUT_DIR / "holdout" / person, loaders)

    rows_h = []
    for name in MODELS_TO_RUN:
        accs = [HOLDOUT_RESULTS[(p, name)]["test_acc"] for p in HOLDOUT_PEOPLE]
        f1s = [HOLDOUT_RESULTS[(p, name)]["test_f1_macro"] for p in HOLDOUT_PEOPLE]
        rows_h.append({"Model": DISPLAY[name],
                       **{f"Akurasi {p}": round(a, 4) for p, a in zip(HOLDOUT_PEOPLE, accs)},
                       "Rata-rata akurasi": round(float(np.mean(accs)), 4),
                       "SD antar-orang": round(float(np.std(accs, ddof=1)), 4) if len(accs) > 1 else None,
                       "Terendah": round(min(accs), 4),
                       "Tertinggi": round(max(accs), 4),
                       "Rata-rata macro-F1": round(float(np.mean(f1s)), 4),
                       "Akurasi sesi dikenal (per gambar)": round(RESULTS[name]["test_acc"], 4)})
    holdout_table = pd.DataFrame(rows_h)
    display(holdout_table)
else:
    print("RUN_PERSON_HOLDOUT = False: bagian ini dilewati.")


## 14. Analisis akhir uji orang baru
Semua angka di bagian ini dihitung dari hasil yang sudah tersimpan, tanpa melatih ulang.

- **Pembanding:** tebakan acak dan 1NN piksel 32x32, dengan data latih yang sama seperti model.
- **Peringkat antar-model:** peringkat per orang dan uji Friedman untuk empat model *transfer learning*. Dengan tiga orang dan satu *seed*, urutan di antara keempatnya tidak dapat disimpulkan.
- **Ketidakpastian:** unit sebenarnya pada uji orang baru adalah orang (n=3), sehingga yang dilaporkan adalah akurasi tiap orang, rata-rata, SD, dan rentang. CI per frame hanya menggambarkan ketidakpastian pengukuran pada foto orang itu.
- **Validasi:** hubungan validasi dengan akurasi orang baru, untuk melihat apakah validasi bisa dipakai memilih model.
- **Faktor pengganggu:** uji orang baru sekaligus menguji sesi, kamera, dan tingkat pembesaran foto. Tabel ukuran kotak tangan memperlihatkan salah satunya; hubungan ini tercampur dengan huruf.
- **Per huruf:** *recall* tiap huruf dan jumlah huruf yang tidak pernah diprediksi.


In [ ]:
from scipy import stats

NN_SIZE = 32


def rgb_vector(relpath):
    img = load_image(DATA_ROOT / relpath).resize((NN_SIZE, NN_SIZE), Image.BILINEAR)
    return np.asarray(img, dtype=np.float32).ravel() / 255


def nearest_neighbor(X_train, y_train, X_test, chunk=256):
    """1NN Euclidean. Jarak kuadrat cukup dihitung sebagai |b|^2 - 2ab karena |a|^2 sama untuk satu baris uji."""
    norms = (X_train ** 2).sum(1)
    return np.concatenate([y_train[(norms - 2 * X_test[i:i + chunk] @ X_train.T).argmin(1)]
                           for i in range(0, len(X_test), chunk)])


def holdout_pred(person, name):
    d = np.load(OUT_DIR / "holdout" / person / f"test_pred_{name}.npz")
    return d["y"], d["p"]


def cluster_ci(hits, groups, n_boot=2000, seed=SEED):
    """CI 95% akurasi dengan mengambil sampel per kelompok, bukan per frame."""
    rng = np.random.default_rng(seed)
    frame = pd.DataFrame({"g": groups, "hit": hits}).groupby("g")["hit"].agg(["sum", "count"]).to_numpy()
    draws = rng.integers(0, len(frame), (n_boot, len(frame)))
    return np.percentile(frame[draws, 0].sum(1) / frame[draws, 1].sum(1), [2.5, 97.5])


TL_MODELS = [n for n in MODELS_TO_RUN if n != "cnn_scratch"]

# Sesi dikenal: CI per kelompok (sumber, huruf)
rows_ci = []
for name in MODELS_TO_RUN:
    y, p = load_pred(name)
    assert (df_test["label_id"].to_numpy() == y).all()
    low, high = cluster_ci(y == p, (df_test["source"] + "_" + df_test["label"]).to_numpy())
    rows_ci.append({"Model": DISPLAY[name], "Akurasi (per gambar)": round(RESULTS[name]["test_acc"], 4),
                    "CI 95% per frame": f"{RESULTS[name]['test_acc_ci_low']:.4f}-{RESULTS[name]['test_acc_ci_high']:.4f}",
                    "CI 95% per (sumber, huruf)": f"{low:.4f}-{high:.4f}"})
print("Sesi yang sudah dikenal: selang kepercayaan per frame dibanding per kelompok")
display(pd.DataFrame(rows_ci))

if RUN_PERSON_HOLDOUT:
    # No. 8: pembanding
    X = np.stack([rgb_vector(r) for r in df["relpath"]])
    y_all, split, source = df["label_id"].to_numpy(), df["split"].to_numpy(), df["source"].to_numpy()
    nn_known = float((nearest_neighbor(X[split == "train"], y_all[split == "train"], X[split == "test"]) == y_all[split == "test"]).mean())
    nn_person = {}
    for person in HOLDOUT_PEOPLE:
        tr_mask = (split == "train") & (source != person)
        nn_person[person] = float((nearest_neighbor(X[tr_mask], y_all[tr_mask], X[source == person]) == y_all[source == person]).mean())

    def baseline_row(label, accs, known):
        vals = [accs[p] for p in HOLDOUT_PEOPLE]
        return {"Model": label, **{f"Akurasi {p}": round(accs[p], 4) for p in HOLDOUT_PEOPLE},
                "Rata-rata akurasi": round(float(np.mean(vals)), 4),
                "SD antar-orang": round(float(np.std(vals, ddof=1)), 4) if len(vals) > 1 else None,
                "Terendah": round(min(vals), 4), "Tertinggi": round(max(vals), 4),
                "Akurasi sesi dikenal (per gambar)": round(known, 4)}

    holdout_final = pd.concat([holdout_table, pd.DataFrame([
        baseline_row("Pembanding: 1NN piksel 32x32", nn_person, nn_known),
        baseline_row("Pembanding: tebakan acak", {p: 1 / N_CLASSES for p in HOLDOUT_PEOPLE}, 1 / N_CLASSES)])], ignore_index=True)
    holdout_final.to_csv(OUT_DIR / "holdout_comparison.csv", index=False)
    print("\nUji orang baru dengan pembanding")
    display(holdout_final)

    acc = pd.DataFrame({DISPLAY[n]: {p: HOLDOUT_RESULTS[(p, n)]["test_acc"] for p in HOLDOUT_PEOPLE} for n in TL_MODELS})

    # No. 3: urutan antar-model transfer learning
    print("Peringkat model transfer learning per orang (1 = tertinggi)")
    display((-acc).rank(axis=1).astype(int).T)
    if len(HOLDOUT_PEOPLE) >= 3:
        fr = stats.friedmanchisquare(*[acc[c].to_numpy() for c in acc.columns])
        print(f"Uji Friedman empat model transfer learning: statistik {fr.statistic:.3f}, p = {fr.pvalue:.3f}")

    gaps = {p: min(HOLDOUT_RESULTS[(p, n)]["test_acc"] for n in TL_MODELS) - HOLDOUT_RESULTS[(p, "cnn_scratch")]["test_acc"]
            for p in HOLDOUT_PEOPLE}
    print("Selisih terkecil transfer learning terhadap CNN dari nol, per orang (poin):",
          {p: round(100 * g, 1) for p, g in gaps.items()})

    print("\nPertimbangan memilih model: ukuran dan latensi, disertai rentang akurasi orang baru")
    display(pd.DataFrame([{"Model": DISPLAY[n], "Ukuran (MB)": round(RESULTS[n]["size_mb"], 1),
                           "Latensi CPU (ms)": round(RESULTS[n]["latency_cpu_ms_median"], 1),
                           "Akurasi orang baru terendah": round(acc[DISPLAY[n]].min(), 4),
                           "Akurasi orang baru tertinggi": round(acc[DISPLAY[n]].max(), 4)} for n in TL_MODELS]))

    # No. 6: validasi dan akurasi orang baru
    pairs = pd.DataFrame([{"person": p, "val": HOLDOUT_RESULTS[(p, n)]["best_val_acc"], "test": HOLDOUT_RESULTS[(p, n)]["test_acc"]}
                          for p in HOLDOUT_PEOPLE for n in TL_MODELS])
    if len(pairs) >= 4:
        rho = stats.spearmanr(pairs["val"], pairs["test"])
        print(f"\nSpearman validasi vs akurasi orang baru ({len(pairs)} pasangan): rho = {rho.statistic:.2f}, p = {rho.pvalue:.2f}")
        print("Per orang:", {p: round(float(stats.spearmanr(g["val"], g["test"]).statistic), 2) for p, g in pairs.groupby("person")})

    # No. 5: ukuran kotak tangan
    def hand_area(relpath):
        with Image.open(DATA_ROOT / relpath) as im:
            rgb = im.convert("RGB")
        box = find_hand_box(rgb)
        return np.nan if box is None else 100 * (box[2] - box[0]) * (box[3] - box[1]) / (rgb.size[0] * rgb.size[1])

    rows_area = []
    for person in HOLDOUT_PEOPLE:
        frame = df[df["source"] == person]
        area = np.array([hand_area(r) for r in frame["relpath"]])
        hits = np.mean([np.equal(*holdout_pred(person, n)) for n in TL_MODELS], axis=0)
        tertile = pd.qcut(area, 3, labels=False, duplicates="drop")
        by = pd.Series(hits).groupby(tertile).mean()
        rows_area.append({"Orang": person, "Luas median kotak tangan (% gambar)": round(float(np.nanmedian(area)), 1),
                          **{f"Akurasi TL, kotak {lab}": round(float(by.get(i, np.nan)), 4) for i, lab in enumerate(["kecil", "sedang", "besar"])}})
    print("\nUkuran kotak tangan pada foto asli dan rata-rata akurasi empat model TL (tercampur dengan huruf)")
    display(pd.DataFrame(rows_area))

    # No. 9: recall per huruf
    recall = {}
    never = {}
    for name in MODELS_TO_RUN:
        rec = []
        for person in HOLDOUT_PEOPLE:
            y, p = holdout_pred(person, name)
            assert (df[df["source"] == person]["label_id"].to_numpy() == y).all()
            rec.append(precision_recall_fscore_support(y, p, labels=list(range(N_CLASSES)), zero_division=0)[1])
            never[(DISPLAY[name], person)] = N_CLASSES - len(np.unique(p))
        recall[DISPLAY[name]] = np.mean(rec, axis=0)
    recall_table = pd.DataFrame(recall, index=CLASSES).round(3)
    recall_table.to_csv(OUT_DIR / "holdout_recall_per_letter.csv")
    print("\nRecall per huruf pada orang baru (rata-rata antar-orang)")
    display(recall_table)
    print("Jumlah huruf yang tidak pernah diprediksi")
    display(pd.Series(never).unstack())


## 15. (Opsional) Uji pada foto buatan sendiri
Letakkan foto Anda di folder dengan struktur yang sama dengan dataset (`A/`, `B/`, ..., `Z/`; boleh hanya sebagian huruf), isi `EXTERNAL_TEST_DIR` pada sel konfigurasi, lalu jalankan sel berikut. Selisih akurasi terhadap data uji dataset adalah indikator generalisasi ke orang, latar, dan kamera yang berbeda.

In [ ]:
def evaluate_external(name, folder):
    root, found = find_class_root(folder, require_all=False)
    rows = []
    for letter, fdir in sorted(found.items()):
        for p in sorted(fdir.rglob("*")):
            if p.is_file() and p.suffix.lower() in IMG_EXT:
                rows.append((str(p.relative_to(root)), letter, CLASSES.index(letter)))
    ext = pd.DataFrame(rows, columns=["relpath", "label", "label_id"])
    print(f"{len(ext)} foto dari {ext['label'].nunique()} huruf")
    ds = ASLDataset(ext, root, eval_tf)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    model = build_model(name, N_CLASSES, pretrained=False).to(device)
    model.load_state_dict(torch.load(CKPT_DIR / f"{name}_best.pt", map_location=device))
    _, acc, p, y = evaluate(model, loader)
    ext["pred"] = [CLASSES[i] for i in p]
    pr, rc, f1 = macro_metrics(y, p)
    print(f"{DISPLAY[name]}: akurasi {acc:.4f}, macro-F1 {f1:.4f}  "
          f"(akurasi data uji dataset: {RESULTS[name]['test_acc']:.4f})")
    return ext


if EXTERNAL_TEST_DIR:
    ext_results = {n: evaluate_external(n, EXTERNAL_TEST_DIR) for n in MODELS_TO_RUN}
    bad = ext_results[BEST][ext_results[BEST]["label"] != ext_results[BEST]["pred"]]
    print(f"\nKesalahan model dengan validasi tertinggi ({DISPLAY[BEST]}):")
    display(bad[["relpath", "label", "pred"]].head(30))
else:
    print("EXTERNAL_TEST_DIR belum diisi: bagian opsional dilewati.")

## Catatan untuk laporan
- **Dua ukuran kinerja:** uji orang baru (bagian 13 dan 14) adalah ukuran generalisasi utama. Akurasi pembagian waktu (bagian 11 dan 12) harus ditulis sebagai "akurasi pada frame lanjutan dari sesi yang sudah dikenal model", bukan sebagai kinerja model secara umum. Identitas partisipan Kaggle tidak tersedia, sehingga hanya ketiga anggota yang dapat ditahan sebagai orang baru.
- **Cara hitung:** akurasi validasi dirata-ratakan per sumber. Bandingkan dengan akurasi uji *rata-rata per sumber*. Akurasi *per gambar* didominasi Kaggle.
- **Perbandingan model:** kesimpulan yang didukung data adalah *transfer learning* lebih baik daripada CNN dari nol pada orang baru. Keempat model *transfer learning* tidak dapat diurutkan dengan tiga orang dan satu *seed*; jika harus memilih, gunakan ukuran dan latensi sambil menyebut rentang akurasinya. Semua kesimpulan berlaku untuk protokol ini.
- **Ketidakpastian:** laporkan akurasi tiap orang, rata-rata, SD, dan rentang (n=3). CI per frame hanya menggambarkan ketidakpastian pengukuran pada foto orang itu, bukan pada populasi.
- **Validasi** tidak dipakai untuk mengklaim model terbaik bagi orang baru, karena validasinya berisi frame lanjutan dari sesi Kaggle dan dua orang lainnya.
- **Faktor pengganggu uji orang baru:** selain orangnya, uji ini juga berbeda sesi, kamera, dan tingkat pembesaran foto, jadi penurunan akurasi tidak seluruhnya efek orang.
- **Pembanding:** selalu sertakan tebakan acak dan 1NN piksel agar akurasi orang baru punya acuan.
- **Batasan lain:** huruf J dan Z aslinya bergerak sedangkan dataset hanya gambar statis; dataset kecil sehingga rawan *overfitting*; sistem yang dipakai adalah ASL (bukan BISINDO/SIBI).
- **Dataset:** catat hasil verifikasi pada bagian 3–5 (jumlah gambar, ukuran, berkas korup, duplikat persis, kelompok nyaris kembar, pasangan lintas kelas, perkiraan jumlah partisipan dan latar secara visual, lisensi di halaman Kaggle).
- **Foto kelompok** di-*crop* dan diperbesar agar sama dengan format Kaggle, sehingga lebih buram dan titik kerangkanya relatif lebih besar. Beberapa foto masih memperlihatkan sebagian wajah; mintalah persetujuan sebelum memakainya sebagai contoh.
- **Reproduksibilitas:** `SEED`, manifes pembagian (`split_manifest.csv`), versi `torch`/`torchvision` yang tercetak di bagian 6, dan seluruh hiperparameter pada bagian 2.
- **Kecepatan inferensi** diukur pada GPU/CPU yang dipakai; sebutkan tipe GPU dan jumlah thread CPU saat melaporkan.
- Data uji tidak dipakai untuk memilih model atau hiperparameter. Hiperparameter sudah dibekukan untuk run ini.
